# DM Analysis Notebook
This notebook analyses sample election results with explanatory markdown between each code cell.

In [ ]:
import json  # Parse JSON result files.
from pathlib import Path  # Work with file-system paths safely.
from statistics import mean  # Compute simple averages for constituency totals.

## Step 1: Locate Input Files
The next code cell discovers all sample result JSON files.

In [3]:
base_path = Path("resources/sample-election-results")  # Point to the sample results directory.
json_files = sorted(base_path.glob("*.json"))  # Collect and sort every JSON result file.
print(f"Files found: {len(json_files)}")  # Show how many files were discovered.
print(json_files[:5])  # Preview the first few paths for validation.

Files found: 650
[PosixPath('resources/sample-election-results/result001.json'), PosixPath('resources/sample-election-results/result002.json'), PosixPath('resources/sample-election-results/result003.json'), PosixPath('resources/sample-election-results/result004.json'), PosixPath('resources/sample-election-results/result005.json')]


## Step 2: Load and Validate Records
The next cell loads each file and keeps records in memory for quick analysis.

In [4]:
records = []  # Store parsed JSON documents here.
for file_path in json_files:  # Iterate through each input file path.
    with file_path.open("r", encoding="utf-8") as handle:  # Open the current file with UTF-8 encoding.
        payload = json.load(handle)  # Parse JSON into a Python dictionary.
    records.append(payload)  # Add parsed data to the in-memory list.
print(f"Records loaded: {len(records)}")  # Confirm total records loaded.
print(records[0].keys() if records else "No records")  # Preview top-level keys when available.

Records loaded: 650
dict_keys(['id', 'name', 'seqNo', 'partyResults'])


## Step 3: Compute Basic Metrics
This cell computes constituency counts and total-vote statistics from party results.

In [9]:
total_votes_per_constituency = [sum(party.get("votes", 0) for party in item.get("partyResults", [])) for item in records]  # Sum party votes for each constituency.
total_constituencies = len(records)  # Count loaded constituency result documents.
average_total_votes = mean(total_votes_per_constituency) if total_votes_per_constituency else 0  # Compute safe average total votes.
max_total_votes = max(total_votes_per_constituency) if total_votes_per_constituency else 0  # Compute safe maximum total votes.
min_total_votes = min(total_votes_per_constituency) if total_votes_per_constituency else 0  # Compute safe minimum total votes.
print(f"Constituencies: {total_constituencies}")  # Print constituency count.
print(f"Average total votes: {average_total_votes:.2f}")  # Print average constituency total votes.
print(f"Max total votes: {max_total_votes:.2f}")  # Print maximum constituency total votes.
print(f"Min total votes: {min_total_votes:.2f}")  # Print minimum constituency total votes.

Constituencies: 650
Average total votes: 41803.12
Max total votes: 66843.00
Min total votes: 14884.00


## Step 4: Quick Party Frequency Scan
The final code cell derives each winner from party vote totals and counts party wins.

In [11]:
party_counts = {}  # Store winner frequency by party name.
for item in records:  # Iterate over each loaded result record.
    party_results = item.get("partyResults", [])  # Read per-party results for this constituency.
    winning_party = max(party_results, key=lambda p: p.get("votes", 0)).get("party", "UNKNOWN") if party_results else "UNKNOWN"  # Derive winner from highest vote count.
    party_counts[winning_party] = party_counts.get(winning_party, 0) + 1  # Increment count for this party.
sorted_counts = sorted(party_counts.items(), key=lambda pair: pair[1], reverse=True)  # Sort party counts in descending order.
print("Top party counts:")  # Print an output heading.
for party, count in sorted_counts[:10]:  # Show the top ten parties by win count.
    print(f"{party}: {count}")  # Print one formatted line per party.

Top party counts:
LAB: 349
CON: 210
LD: 62
DUP: 9
SNP: 6
SF: 5
SDLP: 3
PC: 2
OTH: 1
IKHH: 1
